# 01 · Descarga de datos NOAA Storm Events


Instalo las librerías necesarias para la descarga y lectura de datos


In [1]:
#   %pip install requests beautifulsoup4 pandas

Importo las librerías que voy a utilizar


In [2]:
from pathlib import Path
from urllib.parse import urljoin
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup

Defino la fuente de datos, el período 2010–2025 y la carpeta de trabajo


In [3]:
BASE_URL = (
    "https://www.ncei.noaa.gov/pub/data/"
    "swdi/stormevents/csvfiles/")

# Período
START_YEAR = 2010
END_YEAR = 2025

# Carpeta local
RAW_DIR = Path("../data/raw/noaa/details")

Consulto el directorio público de NOAA y recupero los enlaces disponibles


In [5]:
response = requests.get(BASE_URL, timeout=60)

# Genera un error claro si la página no respondió correctamente
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

links = [
    link.get("href")
    for link in soup.find_all("a", href=True)]

print("Cantidad total de enlaces encontrados:", len(links))

Cantidad total de enlaces encontrados: 238


Identifico los archivos `StormEvents_details` disponibles y extraigo su año y fecha de creación


In [6]:
pattern = re.compile(
    r"^StormEvents_details-ftp_v1\.0_"
    r"d(\d{4})_"
    r"c(\d{8})"
    r"\.csv\.gz$")

available_files = []

for filename in links:
    match = pattern.match(filename)

    if match:
        year = int(match.group(1))
        creation_date = match.group(2)

        available_files.append(
            {
                "year": year,
                "creation_date": creation_date,
                "filename": filename,})

df_available = pd.DataFrame(available_files)

df_available.head()

,year,creation_date,filename
0,1950,20260323,StormEvents_details-ftp_v1.0_d1950_c20260323.c...
1,1951,20260323,StormEvents_details-ftp_v1.0_d1951_c20260323.c...
2,1952,20260323,StormEvents_details-ftp_v1.0_d1952_c20260323.c...
3,1953,20260323,StormEvents_details-ftp_v1.0_d1953_c20260323.c...
4,1954,20260323,StormEvents_details-ftp_v1.0_d1954_c20260323.c...


Selecciono la versión más reciente disponible para cada año entre 2010 y 2025


In [7]:
df_selected = (
    df_available
    .loc[
        df_available["year"].between(
            START_YEAR,
            END_YEAR
        )
    ]
    .sort_values(["year", "creation_date"])
    .drop_duplicates(
        subset="year",
        keep="last"
    )
    .reset_index(drop=True)
)

df_selected

,year,creation_date,filename
0,2010,20260323,StormEvents_details-ftp_v1.0_d2010_c20260323.c...
1,2011,20260323,StormEvents_details-ftp_v1.0_d2011_c20260323.c...
2,2012,20260323,StormEvents_details-ftp_v1.0_d2012_c20260323.c...
3,2013,20260323,StormEvents_details-ftp_v1.0_d2013_c20260323.c...
4,2014,20260323,StormEvents_details-ftp_v1.0_d2014_c20260323.c...
5,2015,20260323,StormEvents_details-ftp_v1.0_d2015_c20260323.c...
6,2016,20260323,StormEvents_details-ftp_v1.0_d2016_c20260323.c...
7,2017,20260519,StormEvents_details-ftp_v1.0_d2017_c20260519.c...
8,2018,20260323,StormEvents_details-ftp_v1.0_d2018_c20260323.c...
9,2019,20260323,StormEvents_details-ftp_v1.0_d2019_c20260323.c...


Compruebo que exista un archivo seleccionado para cada año entre 2010 y 2025


In [8]:
expected_years = set(
    range(START_YEAR, END_YEAR + 1)
)

found_years = set(df_selected["year"])

missing_years = sorted(
    expected_years - found_years
)

print("Archivos seleccionados:", len(df_selected))
print("Años esperados:", len(expected_years))
print("Años faltantes:", missing_years)

Archivos seleccionados: 16
Años esperados: 16
Años faltantes: []


Defino una función para descargar cada archivo sin repetir descargas existentes


In [9]:
def download_file(url, destination):
    """
    Descarga un archivo desde una URL.

    Parámetros
    ----------
    url:
        Dirección web del archivo.

    destination:
        Ruta local donde se guardará.
    """

    # Si ya existe, no vuelve a descargarlo
    if destination.exists():
        print(f"Ya existe: {destination.name}")
        return

    print(f"Descargando: {destination.name}")

    with requests.get(
        url,
        stream=True,
        timeout=180
    ) as response:

        response.raise_for_status()

        with open(destination, "wb") as file:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    file.write(chunk)

    print(f"Descarga finalizada: {destination.name}")

Descargo los archivos seleccionados y los guardo en la carpeta de datos originales


In [10]:
for row in df_selected.itertuples():

    file_url = urljoin(
        BASE_URL,
        row.filename)

    destination = RAW_DIR / row.filename

    download_file(
        url=file_url,
        destination=destination)

Descargando: StormEvents_details-ftp_v1.0_d2010_c20260323.csv.gz
Descarga finalizada: StormEvents_details-ftp_v1.0_d2010_c20260323.csv.gz
Descargando: StormEvents_details-ftp_v1.0_d2011_c20260323.csv.gz
Descarga finalizada: StormEvents_details-ftp_v1.0_d2011_c20260323.csv.gz
Descargando: StormEvents_details-ftp_v1.0_d2012_c20260323.csv.gz
Descarga finalizada: StormEvents_details-ftp_v1.0_d2012_c20260323.csv.gz
Descargando: StormEvents_details-ftp_v1.0_d2013_c20260323.csv.gz
Descarga finalizada: StormEvents_details-ftp_v1.0_d2013_c20260323.csv.gz
Descargando: StormEvents_details-ftp_v1.0_d2014_c20260323.csv.gz
Descarga finalizada: StormEvents_details-ftp_v1.0_d2014_c20260323.csv.gz
Descargando: StormEvents_details-ftp_v1.0_d2015_c20260323.csv.gz
Descarga finalizada: StormEvents_details-ftp_v1.0_d2015_c20260323.csv.gz
Descargando: StormEvents_details-ftp_v1.0_d2016_c20260323.csv.gz
Descarga finalizada: StormEvents_details-ftp_v1.0_d2016_c20260323.csv.gz
Descargando: StormEvents_details-f

Compruebo cuántos archivos se descargaron y cuánto ocupa cada uno


In [11]:
downloaded_files = sorted(RAW_DIR.glob("StormEvents_details-ftp_*.csv.gz"))

print("Cantidad de archivos descargados:",len(downloaded_files))

for file in downloaded_files:
    size_mb = file.stat().st_size / (1024 ** 2)

    print(f"{file.name} - {size_mb:.2f} MB")

Cantidad de archivos descargados: 16
StormEvents_details-ftp_v1.0_d2010_c20260323.csv.gz - 11.19 MB
StormEvents_details-ftp_v1.0_d2011_c20260323.csv.gz - 14.97 MB
StormEvents_details-ftp_v1.0_d2012_c20260323.csv.gz - 11.27 MB
StormEvents_details-ftp_v1.0_d2013_c20260323.csv.gz - 11.18 MB
StormEvents_details-ftp_v1.0_d2014_c20260323.csv.gz - 10.57 MB
StormEvents_details-ftp_v1.0_d2015_c20260323.csv.gz - 9.58 MB
StormEvents_details-ftp_v1.0_d2016_c20260323.csv.gz - 8.65 MB
StormEvents_details-ftp_v1.0_d2017_c20260519.csv.gz - 8.91 MB
StormEvents_details-ftp_v1.0_d2018_c20260323.csv.gz - 9.14 MB
StormEvents_details-ftp_v1.0_d2019_c20260323.csv.gz - 11.07 MB
StormEvents_details-ftp_v1.0_d2020_c20260323.csv.gz - 9.96 MB
StormEvents_details-ftp_v1.0_d2021_c20260323.csv.gz - 10.07 MB
StormEvents_details-ftp_v1.0_d2022_c20260625.csv.gz - 11.27 MB
StormEvents_details-ftp_v1.0_d2023_c20260323.csv.gz - 12.29 MB
StormEvents_details-ftp_v1.0_d2024_c20260728.csv.gz - 12.11 MB
StormEvents_details-ftp

Creo un control de tamaño para detectar archivos vacíos


In [12]:
download_report = []

for file in downloaded_files:

    size_mb = file.stat().st_size / (1024 ** 2)

    download_report.append(
        {
            "filename": file.name,
            "size_mb": round(size_mb, 2),
            "is_empty": file.stat().st_size == 0,})

df_download_report = pd.DataFrame(download_report)

df_download_report

,filename,size_mb,is_empty
0,StormEvents_details-ftp_v1.0_d2010_c20260323.c...,11.19,False
1,StormEvents_details-ftp_v1.0_d2011_c20260323.c...,14.97,False
2,StormEvents_details-ftp_v1.0_d2012_c20260323.c...,11.27,False
3,StormEvents_details-ftp_v1.0_d2013_c20260323.c...,11.18,False
4,StormEvents_details-ftp_v1.0_d2014_c20260323.c...,10.57,False
5,StormEvents_details-ftp_v1.0_d2015_c20260323.c...,9.58,False
6,StormEvents_details-ftp_v1.0_d2016_c20260323.c...,8.65,False
7,StormEvents_details-ftp_v1.0_d2017_c20260519.c...,8.91,False
8,StormEvents_details-ftp_v1.0_d2018_c20260323.c...,9.14,False
9,StormEvents_details-ftp_v1.0_d2019_c20260323.c...,11.07,False


Compruebo si existe algún archivo descargado vacío


In [13]:
df_download_report["is_empty"].value_counts()

is_empty
False    16
Name: count, dtype: int64

Cargo el archivo de 2025 y reviso sus primeras filas


In [14]:
file_2025 = [
    file
    for file in downloaded_files
    if "_d2025_" in file.name][0]

df_2025 = pd.read_csv(
    file_2025,
    low_memory=False)

print("Filas:", df_2025.shape[0])
print("Columnas:", df_2025.shape[1])

df_2025.head()

Filas: 72360
Columnas: 51


,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
0,202503,31,1104,202503,31,1106,201366,1252415,GEORGIA,13,...,2.22,W,TYUS,33.4757,-85.238,33.4757,-85.238,A cold-front initiated a line of thunderstorms...,Tree down at the intersection of highway 5 and...,CSV
1,202503,30,1552,202503,30,1555,200337,1241136,MICHIGAN,26,...,1.47,NNE,EDWARDSBURG,41.7900,-86.100,41.8200,-86.070,A cold front pushed into the area during the a...,A brief EF-1 tornado was confirmed in Edwardsb...,CSV
2,202501,5,1800,202501,6,2227,197733,1222851,VIRGINIA,51,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,An area of low pressure tracked across souther...,NaN,CSV
3,202501,3,1300,202501,3,1900,197761,1223112,MARYLAND,24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,An area of low pressure moved off into New Eng...,NaN,CSV
4,202501,3,1300,202501,3,1900,197761,1223113,MARYLAND,24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,An area of low pressure moved off into New Eng...,NaN,CSV


Reviso las columnas disponibles en el archivo 2025


In [15]:
df_2025.columns.tolist()

['BEGIN_YEARMONTH',
 'BEGIN_DAY',
 'BEGIN_TIME',
 'END_YEARMONTH',
 'END_DAY',
 'END_TIME',
 'EPISODE_ID',
 'EVENT_ID',
 'STATE',
 'STATE_FIPS',
 'YEAR',
 'MONTH_NAME',
 'EVENT_TYPE',
 'CZ_TYPE',
 'CZ_FIPS',
 'CZ_NAME',
 'WFO',
 'BEGIN_DATE_TIME',
 'CZ_TIMEZONE',
 'END_DATE_TIME',
 'INJURIES_DIRECT',
 'INJURIES_INDIRECT',
 'DEATHS_DIRECT',
 'DEATHS_INDIRECT',
 'DAMAGE_PROPERTY',
 'DAMAGE_CROPS',
 'SOURCE',
 'MAGNITUDE',
 'MAGNITUDE_TYPE',
 'FLOOD_CAUSE',
 'CATEGORY',
 'TOR_F_SCALE',
 'TOR_LENGTH',
 'TOR_WIDTH',
 'TOR_OTHER_WFO',
 'TOR_OTHER_CZ_STATE',
 'TOR_OTHER_CZ_FIPS',
 'TOR_OTHER_CZ_NAME',
 'BEGIN_RANGE',
 'BEGIN_AZIMUTH',
 'BEGIN_LOCATION',
 'END_RANGE',
 'END_AZIMUTH',
 'END_LOCATION',
 'BEGIN_LAT',
 'BEGIN_LON',
 'END_LAT',
 'END_LON',
 'EPISODE_NARRATIVE',
 'EVENT_NARRATIVE',
 'DATA_SOURCE']

Normalizo los nombres de las columnas a minúsculas y sin espacios


In [16]:
df_2025.columns = (
    df_2025.columns
    .str.strip()
    .str.lower())

df_2025.columns.tolist()

['begin_yearmonth',
 'begin_day',
 'begin_time',
 'end_yearmonth',
 'end_day',
 'end_time',
 'episode_id',
 'event_id',
 'state',
 'state_fips',
 'year',
 'month_name',
 'event_type',
 'cz_type',
 'cz_fips',
 'cz_name',
 'wfo',
 'begin_date_time',
 'cz_timezone',
 'end_date_time',
 'injuries_direct',
 'injuries_indirect',
 'deaths_direct',
 'deaths_indirect',
 'damage_property',
 'damage_crops',
 'source',
 'magnitude',
 'magnitude_type',
 'flood_cause',
 'category',
 'tor_f_scale',
 'tor_length',
 'tor_width',
 'tor_other_wfo',
 'tor_other_cz_state',
 'tor_other_cz_fips',
 'tor_other_cz_name',
 'begin_range',
 'begin_azimuth',
 'begin_location',
 'end_range',
 'end_azimuth',
 'end_location',
 'begin_lat',
 'begin_lon',
 'end_lat',
 'end_lon',
 'episode_narrative',
 'event_narrative',
 'data_source']

Compruebo los tipos de datos y valores no nulos del archivo 2025


In [17]:
df_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72360 entries, 0 to 72359
Data columns (total 51 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   begin_yearmonth     72360 non-null  int64  
 1   begin_day           72360 non-null  int64  
 2   begin_time          72360 non-null  int64  
 3   end_yearmonth       72360 non-null  int64  
 4   end_day             72360 non-null  int64  
 5   end_time            72360 non-null  int64  
 6   episode_id          72360 non-null  int64  
 7   event_id            72360 non-null  int64  
 8   state               72360 non-null  object 
 9   state_fips          72360 non-null  int64  
 10  year                72360 non-null  int64  
 11  month_name          72360 non-null  object 
 12  event_type          72360 non-null  object 
 13  cz_type             72360 non-null  object 
 14  cz_fips             72360 non-null  int64  
 15  cz_name             72360 non-null  object 
 16  wfo 

Reviso una muestra de variables relevantes del archivo 2025


In [18]:
columns_preview = [
    "event_id",
    "state",
    "year",
    "month_name",
    "event_type",
    "begin_date_time",
    "end_date_time",
    "damage_property",
    "damage_crops",
    "magnitude",]

df_2025[columns_preview].head(10)

,event_id,state,year,month_name,event_type,begin_date_time,end_date_time,damage_property,damage_crops,magnitude
0,1252415,GEORGIA,2025,March,Thunderstorm Wind,31-MAR-25 11:04:00,31-MAR-25 11:06:00,1.00K,NaN,52.0
1,1241136,MICHIGAN,2025,March,Tornado,30-MAR-25 15:52:00,30-MAR-25 15:55:00,100.00K,0.00K,NaN
2,1222851,VIRGINIA,2025,January,Winter Storm,05-JAN-25 18:00:00,06-JAN-25 22:27:00,NaN,NaN,NaN
3,1223112,MARYLAND,2025,January,Winter Weather,03-JAN-25 13:00:00,03-JAN-25 19:00:00,NaN,NaN,NaN
4,1223113,MARYLAND,2025,January,Winter Weather,03-JAN-25 13:00:00,03-JAN-25 19:00:00,NaN,NaN,NaN
5,1223114,MARYLAND,2025,January,Winter Weather,03-JAN-25 13:00:00,03-JAN-25 19:00:00,NaN,NaN,NaN
6,1223150,MARYLAND,2025,January,Winter Weather,03-JAN-25 15:47:00,03-JAN-25 16:19:00,0.00K,0.00K,NaN
7,1223144,VIRGINIA,2025,January,Winter Weather,03-JAN-25 15:27:00,03-JAN-25 16:19:00,NaN,NaN,NaN
8,1223110,MARYLAND,2025,January,Winter Weather,03-JAN-25 13:00:00,03-JAN-25 19:00:00,NaN,NaN,NaN
9,1223111,MARYLAND,2025,January,Winter Weather,03-JAN-25 13:00:00,03-JAN-25 19:00:00,NaN,NaN,NaN
